In [1]:
import pandas as pd
import numpy as np
import re

In [2]:
df = pd.read_excel('Oneplus.xlsx')

In [3]:
df['MRP'] = df['MRP'].str.replace('₹', '').str.replace(' ', '').str.replace(',', '').astype(float)
df['Current Price'] = df['Current Price'].str.replace('₹', '').str.replace(',', '').astype(float)

In [4]:
def extract_discount(value):
    if pd.isna(value):
        return 0  # or return None
    return int(value.replace('% off', ''))

df['Discount_Number'] = df['Discount'].apply(extract_discount)

In [5]:
df.drop("Discount", axis=1, inplace=True)

In [6]:
brands = {
    'Apple': ['apple', 'iphone', 'ipad'],
    'motorola': ['motorola', 'moto'],
    'AI_plus': ['ai plus', 'aiplus'],
    'IQOO': ['iqoo'],
    'Tecno': ['tecno'],
    'lava': ['lava'],
    'readmi': ['readmi', 'redmi'],  
    'realme': ['realme'],
    'POCO': ['poco'],
    'nothing': ['nothing'],
    'Infix': ['infix'],
    'Oppo': ['oppo'],
    'vivo': ['vivo'],
    'Google': ['google', 'pixel'],
    'OnePlus': ['oneplus', 'one plus'],
    'Samsung': ['samsung', 'galaxy'],
}

def get_brand(product_name):
    if pd.isna(product_name):
        return 'Unknown'
    
    product_name = str(product_name).lower()
    
    for brand, keywords in brands.items():
        for keyword in keywords:
            if keyword in product_name:
                return brand
    return 'Unknown'
df['Brand'] = df['Product Name'].apply(get_brand)

In [7]:
df['RAM'] = df['RAM'].str.extract(r'(\d+)').astype(int)

In [8]:
df['Storage'] = df['Storage'].str.extract(r'(\d+)').astype(int)

In [9]:
def extract_ratings_reviews(text):
    """
    Extract ratings and reviews from text
    Example: "2,46,375 Ratings & 9,265 Reviews"
    Returns: (ratings, reviews)
    """
    if pd.isna(text):
        return None, None
    
    text = str(text)
    
    # Extract ratings (numbers before 'Ratings')
    ratings_match = re.search(r'([\d,]+)\s*Ratings', text, re.IGNORECASE)
    ratings = ratings_match.group(1).replace(',', '') if ratings_match else None
    
    # Extract reviews (numbers before 'Reviews')
    reviews_match = re.search(r'([\d,]+)\s*Reviews', text, re.IGNORECASE)
    reviews = reviews_match.group(1).replace(',', '') if reviews_match else None
    
    return ratings, reviews

# Apply to dataframe
df[['Ratings', 'Reviews']] = df['Review text'].apply(
    lambda x: pd.Series(extract_ratings_reviews(x))
)

# Convert to numbers
df['Ratings'] = pd.to_numeric(df['Ratings'], errors='coerce')
df['Reviews'] = pd.to_numeric(df['Reviews'], errors='coerce')

In [10]:
def extract_battery(text):
    if pd.isna(text):
        return 0
    match = re.search(r'(\d+)', str(text))
    return int(match.group()) if match else 0

df['Battery'] = df['Battery'].apply(extract_battery)

In [11]:
df.head()

,Product Name,Current Price,MRP,Rating,Review text,RAM,Storage,Display,Camera,Battery,Processor,Product URL,Image URL,Discount_Number,Brand,Ratings,Reviews
0,"OnePlus Nord CE6 Lite (Hyper Black, 128 GB)",25357.0,31999.0,4.3,436 Ratings & 27 Reviews,6,128,17.07 cm (6.72 inch) Full HD+ Display,50MP Rear Camera | 8MP Front Camera,7000,MediaTek Dimensity 7400 Apex Processor,https://www.flipkart.com/oneplus-nord-ce6-lite...,https://rukminim2.flixcart.com/image/312/312/x...,20,OnePlus,436.0,27.0
1,"OnePlus Nord CE6 Lite (Hyper Black, 128 GB)",28000.0,33999.0,4.3,211 Ratings & 14 Reviews,8,128,17.07 cm (6.72 inch) Full HD+ Display,50MP Rear Camera | 8MP Front Camera,7000,MediaTek Dimensity 7400 Apex Processor,https://www.flipkart.com/oneplus-nord-ce6-lite...,https://rukminim2.flixcart.com/image/312/312/x...,17,OnePlus,211.0,14.0
2,"OnePlus Nord CE6 (Pitch Black, 128 GB)",33190.0,40999.0,4.5,223 Ratings & 13 Reviews,8,128,17.22 cm (6.78 inch) Full HD+ Display,50MP Rear Camera | 2MP Front Camera,8000,Qualcomm® Snapdragon™ 7s Gen 4 Processor,https://www.flipkart.com/oneplus-nord-ce6-pitc...,https://rukminim2.flixcart.com/image/312/312/x...,19,OnePlus,223.0,13.0
3,"OnePlus Nord CE6 Lite (Vivid Mint, 128 GB)",25723.0,31999.0,4.3,436 Ratings & 27 Reviews,6,128,17.07 cm (6.72 inch) Full HD+ Display,50MP Rear Camera | 8MP Front Camera,7000,MediaTek Dimensity 7400 Apex Processor,https://www.flipkart.com/oneplus-nord-ce6-lite...,https://rukminim2.flixcart.com/image/312/312/x...,19,OnePlus,436.0,27.0
4,"OnePlus Nord CE6 Lite (Vivid Mint, 128 GB)",26193.0,33999.0,4.3,211 Ratings & 14 Reviews,8,128,17.07 cm (6.72 inch) Full HD+ Display,8MP Rear Camera | 8MP Front Camera,7000,MediaTek Dimensity 7400 Apex Processor,https://www.flipkart.com/oneplus-nord-ce6-lite...,https://rukminim2.flixcart.com/image/312/312/x...,22,OnePlus,211.0,14.0


In [12]:
df.isnull().sum()

Product Name         0
Current Price        1
MRP                 16
Rating               1
Review text          1
RAM                  0
Storage              0
Display              0
Camera               2
Battery              0
Processor          194
Product URL          0
Image URL            0
Discount_Number      0
Brand                0
Ratings              1
Reviews              1
dtype: int64

In [13]:
df['MRP'] = np.where(
    (df['Discount_Number'] == 0) & (df['MRP'].isna() | (df['MRP'] == '')),
    df['Current Price'],
    df['MRP']
)

In [14]:
df.isnull().sum()

Product Name         0
Current Price        1
MRP                  1
Rating               1
Review text          1
RAM                  0
Storage              0
Display              0
Camera               2
Battery              0
Processor          194
Product URL          0
Image URL            0
Discount_Number      0
Brand                0
Ratings              1
Reviews              1
dtype: int64

In [15]:
def get_oneplus_camera(product_name):
    if pd.isna(product_name):
        return 'Unknown Camera'
    
    # OnePlus 12 Series
    if 'OnePlus 12' in product_name:
        return '50MP + 48MP + 64MP'
    
    # OnePlus 11 Series
    elif 'OnePlus 11' in product_name:
        return '50MP + 48MP + 32MP'
    
    # OnePlus 10 Series
    elif 'OnePlus 10' in product_name:
        if 'Pro' in product_name:
            return '48MP + 50MP + 8MP'
        else:
            return '50MP + 16MP + 2MP'
    
    # OnePlus 9 Series
    elif 'OnePlus 9' in product_name:
        if 'Pro' in product_name:
            return '48MP + 50MP + 8MP + 2MP'
        else:
            return '48MP + 50MP + 2MP'
    
    # OnePlus 8 Series
    elif 'OnePlus 8' in product_name:
        if 'Pro' in product_name:
            return '48MP + 48MP + 8MP + 5MP'
        else:
            return '48MP + 16MP + 2MP'
    
    # OnePlus 7 Series
    elif 'OnePlus 7' in product_name:
        if 'Pro' in product_name:
            return '48MP + 16MP + 8MP'
        else:
            return '48MP + 5MP'
    
    # OnePlus 6 Series
    elif 'OnePlus 6' in product_name:
        return '16MP + 20MP'
    
    # OnePlus Nord Series
    elif 'OnePlus Nord' in product_name:
        return '50MP + 8MP + 2MP'
    
    else:
        return 'Unknown Camera'

df.loc[df['Camera'].isnull(), 'Camera'] = df.loc[df['Camera'].isnull(), 'Product Name'].apply(get_oneplus_camera)

In [16]:
df.isnull().sum()

Product Name         0
Current Price        1
MRP                  1
Rating               1
Review text          1
RAM                  0
Storage              0
Display              0
Camera               0
Battery              0
Processor          194
Product URL          0
Image URL            0
Discount_Number      0
Brand                0
Ratings              1
Reviews              1
dtype: int64

In [17]:
def get_oneplus_processor(product_name):
    if pd.isna(product_name):
        return 'Unknown Processor'
    
    # OnePlus 12 Series
    if 'OnePlus 12' in product_name:
        return 'Snapdragon 8 Gen 3'
    
    # OnePlus 11 Series
    elif 'OnePlus 11' in product_name:
        return 'Snapdragon 8 Gen 2'
    
    # OnePlus 10 Series
    elif 'OnePlus 10' in product_name:
        if 'Pro' in product_name:
            return 'Snapdragon 8 Gen 1'
        else:
            return 'Snapdragon 8 Gen 1'
    
    # OnePlus 9 Series
    elif 'OnePlus 9' in product_name:
        if 'Pro' in product_name:
            return 'Snapdragon 888'
        else:
            return 'Snapdragon 888'
    
    # OnePlus 8 Series
    elif 'OnePlus 8' in product_name:
        if 'Pro' in product_name:
            return 'Snapdragon 865'
        else:
            return 'Snapdragon 865'
    
    # OnePlus 7 Series
    elif 'OnePlus 7' in product_name:
        if 'Pro' in product_name:
            return 'Snapdragon 855'
        else:
            return 'Snapdragon 855'
    
    # OnePlus 6 Series
    elif 'OnePlus 6' in product_name:
        return 'Snapdragon 845'
    
    # OnePlus 5 Series
    elif 'OnePlus 5' in product_name:
        return 'Snapdragon 835'
    
    # OnePlus Nord Series
    elif 'OnePlus Nord' in product_name:
        if 'Nord 4' in product_name or 'Nord CE 4' in product_name:
            return 'Snapdragon 7 Gen 3'
        elif 'Nord 3' in product_name:
            return 'MediaTek Dimensity 9000'
        elif 'Nord 2' in product_name:
            return 'MediaTek Dimensity 1200'
        elif 'Nord CE 3' in product_name:
            return 'Snapdragon 782G'
        elif 'Nord CE 2' in product_name:
            return 'MediaTek Dimensity 900'
        else:
            return 'Snapdragon 765G'
    
    else:
        return 'Unknown Processor'

df.loc[df['Processor'].isnull(), 'Processor'] = df.loc[df['Processor'].isnull(), 'Product Name'].apply(get_oneplus_processor)

In [18]:
df.isnull().sum()

Product Name       0
Current Price      1
MRP                1
Rating             1
Review text        1
RAM                0
Storage            0
Display            0
Camera             0
Battery            0
Processor          0
Product URL        0
Image URL          0
Discount_Number    0
Brand              0
Ratings            1
Reviews            1
dtype: int64

In [19]:
df['Rating'] = df['Rating'].fillna(df['Rating'].median())
df['Ratings'] = df['Ratings'].fillna(df['Ratings'].median()).astype(int)
df['Reviews'] = df['Reviews'].fillna(df['Reviews'].median()).astype(int)

In [20]:
df.isnull().sum()

Product Name       0
Current Price      1
MRP                1
Rating             0
Review text        1
RAM                0
Storage            0
Display            0
Camera             0
Battery            0
Processor          0
Product URL        0
Image URL          0
Discount_Number    0
Brand              0
Ratings            0
Reviews            0
dtype: int64

In [21]:
# Extract the series from product name
def get_oneplus_model(product_name):
    if pd.isna(product_name):
        return None
    # Extract OnePlus model (e.g., "OnePlus 12", "OnePlus 9 Pro")
    match = pd.Series(product_name).str.extract(r'(OnePlus \d+[^\s]*)').iloc[0,0]
    return match

df['OnePlus_Model'] = df['Product Name'].apply(get_oneplus_model)

# Fill the null price with median of same model
for idx in df[df['Current Price'].isnull()].index:
    model = df.loc[idx, 'OnePlus_Model']
    if model:
        median_price = df[df['OnePlus_Model'] == model]['Current Price'].median()
        if not pd.isna(median_price):
            df.loc[idx, 'Current Price'] = median_price
            df.loc[idx, 'MRP'] = df.loc[idx, 'MRP'] if not pd.isna(df.loc[idx, 'MRP']) else median_price
        else:
            # Fallback to overall median
            df.loc[idx, 'Current Price'] = df['Current Price'].median()
            df.loc[idx, 'MRP'] = df['Current Price'].median()

# Remove temporary column
df = df.drop('OnePlus_Model', axis=1)

In [22]:
df.isnull().sum()

Product Name       0
Current Price      0
MRP                0
Rating             0
Review text        1
RAM                0
Storage            0
Display            0
Camera             0
Battery            0
Processor          0
Product URL        0
Image URL          0
Discount_Number    0
Brand              0
Ratings            0
Reviews            0
dtype: int64

In [23]:
df.to_excel('Cleaned_Oneplus.xlsx', index=False)